# Cell 1 - Data Loader 

In [ ]:
import os
import pandas as pd
import numpy as np
import shutil

(21375, 35)
CSV filename: XC134874.mp3
Actual files: ['XC78890.mp3', 'XC142067.mp3', 'XC370826.mp3', 'XC247799.mp3', 'XC189264.mp3']


In [ ]:

random_seed = 42

DATA_DIR = "/kaggle/input/competitions/birdsong-recognition"
AUDIO_DIR = os.path.join(DATA_DIR, "train_audio")

train_csv = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
print(train_csv.shape)

# sanity check filenames match what's on disk
sample_species = train_csv.ebird_code.iloc[0]
print("CSV filename:", train_csv[train_csv.ebird_code == sample_species].filename.iloc[0])
print("Actual files:", os.listdir(os.path.join(AUDIO_DIR, sample_species))[:5])

# Cell 2 - Per Species Stats (EDA)

In [2]:
counts = train_csv.groupby("ebird_code").agg(
    n_clips=("ebird_code", "count"),
    avg_duration=("duration", "mean"),
    min_duration=("duration", "min"),
    max_duration=("duration", "max"),
    avg_rating=("rating", "mean")
).reset_index()

print(counts.describe())

          n_clips  avg_duration  min_duration  max_duration  avg_rating
count  264.000000    264.000000    264.000000    264.000000  264.000000
mean    80.965909     53.533975      2.170455    369.397727    3.976190
std     25.776953     20.163487      1.676645    250.355999    0.535230
min      9.000000     17.551724      0.000000     71.000000    1.822581
25%     66.000000     39.595000      1.000000    235.750000    3.668640
50%    100.000000     51.590000      2.000000    320.500000    4.002500
75%    100.000000     64.185000      3.000000    441.000000    4.365000
max    100.000000    137.280000     12.000000   2283.000000    5.000000


# Cell 3 - Filtering Based on maximum clip duration 

In [3]:
MIN_CLIPS = 45
MIN_RATING = 3.5

candidates = counts[
    (counts.n_clips >= MIN_CLIPS) &
    (counts.avg_rating >= MIN_RATING)
].sort_values(["n_clips", "avg_rating"], ascending=[False, False])

print(f"Species passing filter: {len(candidates)}")
print(candidates.head(20))

Species passing filter: 201
    ebird_code  n_clips  avg_duration  min_duration  max_duration  avg_rating
18      barswa      100         48.93             5           288       5.000
77      comrav      100         85.93             2          2283       5.000
128     houwre      100         48.06             1           267       5.000
190     redcro      100         37.49             0           248       5.000
221     sonspa      100         51.58             3           279       5.000
127     houspa      100         48.24             5           329       4.985
224     spotow      100         46.42             2           374       4.950
196     rewbla      100         65.06             1           510       4.935
8       amerob      100         68.12             3           441       4.910
61      carwre      100         54.05             1           224       4.910
157     norcar      100         63.81             3           576       4.860
79      comter      100         51.6

# Cell 4 - Selected 6 species

In [4]:
# Chosen for: common well-known North American species, full clip availability (100),
# high community-rated recording quality (avg_rating > 4.5), and taxonomic/vocalization
# diversity (corvid, thrush, swallow, sparrow-family, wren, flycatcher)
CHOSEN_SPECIES = ["amecro", "amerob", "astfly", "barswa", "bewwre", "eastow"]

chosen_summary = counts[counts.ebird_code.isin(CHOSEN_SPECIES)].sort_values("ebird_code")
print(chosen_summary)

   ebird_code  n_clips  avg_duration  min_duration  max_duration  avg_rating
3      amecro      100         68.48             1           427        4.79
8      amerob      100         68.12             3           441        4.91
13     astfly      100         85.12             3           357        4.54
18     barswa      100         48.93             5           288        5.00
22     bewwre      100         74.81             2           557        4.79
93     eastow      100         57.88             5           329        4.70


# Cell 5 - Exact clips per class (based on duration)

In [ ]:
N_TRAIN_POOL = 25
N_EVAL = 50

curated_rows = []
for species in CHOSEN_SPECIES:
    sub = train_csv[train_csv.ebird_code == species].copy()
    sub = sub.sample(frac=1, random_state=random_seed).reset_index(drop=True)
    
    n_needed = N_TRAIN_POOL + N_EVAL
    sub = sub.iloc[:n_needed].copy()
    sub["split"] = ["train_pool"] * N_TRAIN_POOL + ["eval"] * N_EVAL
    curated_rows.append(sub)

curated_df = pd.concat(curated_rows, ignore_index=True)
curated_df = curated_df.drop(columns=["species"])
curated_df = curated_df.rename(columns={"ebird_code": "species"})

print(curated_df.split.value_counts())
print(curated_df.species.value_counts())

split
train_pool    150
eval          150
Name: count, dtype: int64
species
amecro    50
amerob    50
astfly    50
barswa    50
bewwre    50
eastow    50
Name: count, dtype: int64


In [6]:
# # how many curated clips would need padding (i.e. are shorter than candidate thresholds)?
# for t in [3, 5, 8, 10]:
#     n_short = (curated_df.duration < t).sum()
#     pct_short = 100 * n_short / len(curated_df)
#     print(f"Threshold {t}s: {n_short}/{len(curated_df)} clips ({pct_short:.1f}%) would need padding")

# Cell 6 - Zero padding/ Truncating Clips and Making the dataset

In [7]:
from pydub import AudioSegment

OUT_DIR = "/kaggle/working/curated_dataset"
CLIP_DURATION_SEC = 5

for split in ["train_pool", "eval"]:
    for species in CHOSEN_SPECIES:
        os.makedirs(os.path.join(OUT_DIR, split, species), exist_ok=True)

failed = []
for _, row in curated_df.iterrows():
    src = os.path.join(AUDIO_DIR, row.species, row.filename)
    dst = os.path.join(OUT_DIR, row.split, row.species, row.filename)
    
    try:
        audio = AudioSegment.from_mp3(src)
        target_ms = CLIP_DURATION_SEC * 1000
        
        if len(audio) < target_ms:
            silence = AudioSegment.silent(duration=target_ms - len(audio))
            audio = audio + silence
        else:
            audio = audio[:target_ms]
        
        audio.export(dst, format="mp3")
    except Exception as e:
        failed.append((row.species, row.filename, str(e)))

print(f"Done. {len(curated_df) - len(failed)} succeeded, {len(failed)} failed.")
if failed:
    print(failed[:10])

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Done. 300 succeeded, 0 failed.


# Cell 7 - Final Zip file for the dataset

In [8]:
curated_df[["filename", "species", "split", "duration", "rating"]].to_csv(
    os.path.join(OUT_DIR, "manifest.csv"), index=False
)

shutil.make_archive("/kaggle/working/curated_dataset", "zip", OUT_DIR)
print("Zipped: /kaggle/working/curated_dataset.zip")

Zipped: /kaggle/working/curated_dataset.zip
